# Generator LoRA: 답변 모델(Qwen2.5-7B)을 RAG 형식으로 학습

검색 쪽(Dense·Reranker)은 파인튜닝 효과가 없어 원래 모델을 씁니다 (`FineTuneRetriever.ipynb`). 이 노트북은 **답변을 쓰는 LLM**을 학습합니다.

```
train 의료 질문 → URL DB에서 검색 (BM25 + Dense → RRF → Rerank) → 거절 게이트 통과 → 문서 [1]~[5]
   → 정답 내용이 문서에 있으면 "train 정답 [번호]",  없으면 "거절 문장"  → Qwen2.5-7B에 LoRA 학습
```

| | 데이터 |
|---|---|
| 학습 | **train** 질문만. 거절 기준값을 정한 튜닝 세트와, validation과 문장이 같은 질문은 뺌 |
| 평가 | **validation** 시험지 (`LabeledBaseline.ipynb`와 같음): `lab_med_test` 598, `lab_med_ext_test` 210, `lab_ood_test` 1,000 |
| 비교 | 원본 Qwen vs LoRA Qwen. 둘 다 같은 4bit 모델(MLX)·같은 검색·같은 거절 기준·같은 프롬프트(인용 표시) |

**답할지 거절할지 정하는 기준**: MS MARCO는 질문마다 정답 문서를 1개 정도만 표시합니다. 표시된 문서가 상위 5개에 없어도 다른 문서에 같은 답이 있는 경우가 많아서,
표시만 보고 거절 예시를 만들면 "답할 수 있는데 거절하라"고 가르치게 됩니다. 그래서 **정답 문장의 단어가 문서 5개에 실제로 있는지**로 정합니다.
- 정답 표시 문서가 상위 5개에 있음 → 답변 (그 문서 번호를 인용)
- 표시 문서는 없지만 정답 단어의 80% 이상이 문서 5개에 있음 → 답변 (정답 단어가 들어 있는 문서 번호를 인용)
- 둘 다 아님 → 거절
- train 비의료 질문(`lab_ood_test`와 같은 정의: 정답이 의료 사이트에 없고 라벨 C) 중 거절 게이트를 통과한 것 → 거절
  - 정답이 의료 사이트에 있는 C 질문은 쓰지 않습니다. 의료 문서와 잘 맞아 게이트를 쉽게 통과하고, LLM 라벨이 실제로는 의료 질문인 경우도 섞여 있어 거절 예시가 너무 많아집니다 (처음 시도에서 전체의 31%).

**런타임**: Mac 전용 (Apple Silicon, MLX). 데이터 준비 10분 안팎, 학습 300단계 약 1시간 10분, 평가 2~3시간 (M1 Pro 16GB 기준).

> 학습 기록: 학습률 1e-4는 60~80단계, 2e-5는 301~320단계에서 loss가 `nan`이 됨 (MLX lora에는 gradient clipping이 없어, 한 묶음에서 계산이 터지면 그대로 망가짐). 그래서 nan 직전 저장본인 **300단계(예시 600개, 검증 loss 2.687 → 0.391)**를 최종 LoRA로 씁니다. 실패한 기록은 `results/lora/train_log_lr1e-4_nan.txt`.
학습 중간 결과(`--save-every`)와 생성 결과(`results/lora/gen_cache_mlx.jsonl`)를 저장하므로, 중간에 멈춰도 다시 실행하면 이어서 진행합니다.

## 0. 환경 설정

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:   # 로컬에서는 미리 설치한 패키지를 그대로 씀
    %pip install -q -U datasets bm25s PyStemmer faiss-cpu sentence-transformers rouge-score accelerate bitsandbytes

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
import os, re, json, random, hashlib
from pathlib import Path
from collections import defaultdict
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import torch

# ===== 경로 =====
# Colab은 Drive의 RAG-Chat, 로컬은 RAG-Chat 폴더 (RAG-Chat 또는 RAG-Chat/Baseline 어디서 열어도 찾음)
LOCAL_DIR = Path.cwd() if (Path.cwd() / "Baseline").exists() else Path.cwd().parent
BASE_DIR = Path(os.environ.get("RAG_BASE_DIR", "/content/drive/MyDrive/RAG-Chat" if IN_COLAB else LOCAL_DIR)).resolve()
DATA_DIR = BASE_DIR / "data" / "processed"
INDEX_DIR = BASE_DIR / "index"
RESULT_DIR = BASE_DIR / "results" / "labeled"
URL_RESULT_DIR = BASE_DIR / "results" / "baseline"     # MedicalBaseline 결과 (참고용)
LABEL_PATH = BASE_DIR / "data" / "labels" / "query_labels.parquet"
for d in (DATA_DIR, INDEX_DIR, RESULT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ===== 실험 설정 =====
SEED = 42
DATASET_NAME, DATASET_VERSION = "microsoft/ms_marco", "v1.1"

N_OOD_TEST = 1000      # validation 도메인 밖 질문 수
N_TUNE = 500           # train 튜닝 세트 (의료 / 도메인 밖 각각)

EMBED_MODEL = "BAAI/bge-base-en-v1.5"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "  # bge 검색용 쿼리 instruction
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
GEN_MODEL = "Qwen/Qwen2.5-7B-Instruct"

K_CAND = 100           # BM25 / Dense 후보 수
K_RERANK = 50          # Hybrid 상위 몇 개를 rerank할지
K_EVAL = 10            # Recall@10, MRR@10
K_CTX = 5              # LLM에 넣을 passage 수
RRF_K = 60
MAX_NEW_TOKENS = 128
GEN_BATCH = 16
GEN_CONFIGS = ["No-Retrieval", "Hybrid+Rerank"]   # 다른 검색 방식도 비교하려면 추가

REFUSAL = "I cannot answer this question based on the provided documents."

# ===== 장치와 생성 방식 =====
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
USE_FP16 = DEVICE == "cuda"
GEN_BACKEND = "hf" if DEVICE == "cuda" else "ollama"   # GPU가 없으면 Ollama로 생성
OLLAMA_URL = "http://localhost:11434"
OLLAMA_MODEL = "qwen2.5:7b-instruct"   # Qwen/Qwen2.5-7B-Instruct의 Q4_K_M 양자화
OLLAMA_WORKERS = 4                     # Ollama 동시 요청 수
print("device:", DEVICE, "/ 생성:", GEN_BACKEND, "/ BASE_DIR:", BASE_DIR)

random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# ===== LoRA 설정 =====
MLX_MODEL = "mlx-community/Qwen2.5-7B-Instruct-4bit"   # Qwen/Qwen2.5-7B-Instruct의 4bit (MLX)
GEN_DB = "url"         # 학습·평가 모두 URL DB에서 검색 (시스템 비교에서 가장 좋았던 DB)
N_TRAIN = 2000         # 학습 예시 수
N_VALID = 100          # 학습 중 loss 확인용 (train에서 따로 뗌, 평가에는 안 씀)
ANSWER_IN_CTX = 0.8    # 정답 단어 중 이만큼이 문서 5개에 있으면 답할 수 있는 질문
CITE_PASSAGE_MIN = 0.5 # 정답 단어 중 이만큼이 들어 있는 문서를 인용 번호로 씀
LORA_BATCH, LORA_GRAD_ACC = 2, 4   # 실제 배치 8 (메모리 때문에 2개씩 4번 나눠 계산)
LORA_LR = 2e-5          # 1e-4에서는 60~80단계에서 loss가 nan이 됨 (MLX lora에는 gradient clipping이 없음)
LORA_LAYERS = 16       # 위쪽 16개 층에 LoRA
LORA_MAX_LEN = 1536
# 1 epoch(1,000단계)로 시작했지만 2e-5에서도 301~320단계에서 loss가 nan이 됨 (1e-4에서는 60~80단계).
# 그 전 저장본(300단계 = 예시 600개)을 최종으로 씀. loss는 100단계 이후 0.4~0.6에서 거의 그대로였음
LORA_ITERS = 300
EVAL_BATCH = 8         # 평가 생성 배치
RETRAIN = False        # False면 저장된 LoRA가 있을 때 학습을 건너뜀

LORA_DATA_DIR = DATA_DIR / "lora"
LORA_DIR = BASE_DIR / "models" / "qwen2.5-7b-instruct-rag-lora"
RESULT_DIR = BASE_DIR / "results" / "lora"
for d in (LORA_DATA_DIR, LORA_DIR, RESULT_DIR):
    d.mkdir(parents=True, exist_ok=True)
assert not IN_COLAB, "이 노트북은 Mac(MLX) 전용입니다"

## 1. 데이터 준비 (LabeledBaseline과 동일 코드)

In [ ]:
MEDICAL_DOMAINS = [
    "webmd.com", "webmd.boots.com", "mayoclinic.org", "mayoclinic.com", "medicinenet.com",
    "healthline.com", "medical-dictionary.thefreedictionary.com", "nih.gov", "medlineplus.gov",
    "cdc.gov", "everydayhealth.com", "nhs.uk", "medicalnewstoday.com", "rxlist.com",
    "medscape.com", "emedicinehealth.com", "wisegeekhealth.com", "health.howstuffworks.com",
    "cancer.org", "clevelandclinic.org", "patient.info", "hopkinsmedicine.org", "healthtap.com",
    "news-medical.net", "cancerresearchuk.org", "americanpregnancy.org", "spine-health.com",
    "who.int", "cancer.gov", "womenshealth.gov", "md-health.com", "heart.org", "healthhype.com",
    "familydoctor.org", "goodrx.com", "medhelp.org", "symptomfind.com", "medilexicon.com",
    "psychcentral.com", "healthcommunities.com", "med-health.net", "diabetes.org",
    "themedicalquestions.com", "health.harvard.edu", "drugs.com", "cancercenter.com",
    "rightdiagnosis.com", "innerbody.com", "labtestsonline.org", "sharecare.com", "umm.edu",
    "kidshealth.org", "health.com", "healthguideinfo.com", "medicalhealthtests.com",
]

def get_host(url):
    host = (urlparse(url).hostname or "").lower()
    return host[4:] if host.startswith("www.") else host

def is_medical_host(host):
    return any(host == d or host.endswith("." + d) for d in MEDICAL_DOMAINS)

def normalize_text(text):
    return re.sub(r"\s+", " ", str(text)).strip()

def has_answer(answers):
    return any(
        str(a).strip() and str(a).strip().lower().rstrip(".") != "no answer present"
        for a in (answers if answers is not None else [])
    )

In [ ]:
from datasets import load_dataset

dataset = load_dataset(DATASET_NAME, name=DATASET_VERSION)
qa = {split: dataset[split].to_pandas() for split in ("train", "validation")}
for split, df in qa.items():
    print(split, len(df))

In [ ]:
rows = []
for split, df in qa.items():
    for qid, p in zip(df["query_id"], df["passages"]):
        for text, sel, url in zip(p["passage_text"], p["is_selected"], p["url"]):
            rows.append((split, int(qid), normalize_text(text), int(sel), url))

flat = pd.DataFrame(rows, columns=["split", "query_id", "text", "is_selected", "url"])
flat = flat[flat["text"] != ""]
flat["host"] = flat["url"].map(get_host)
host_is_med = {h: is_medical_host(h) for h in flat["host"].unique()}
flat["is_medical"] = flat["host"].map(host_is_med)

corpus = (
    flat[flat["is_medical"]]
    .drop_duplicates("text")[["text", "url", "host"]]
    .reset_index(drop=True)
)
corpus.insert(0, "pid", np.arange(len(corpus)))   # pid == 행 번호 (인덱스 위치와 동일)
text2pid = dict(zip(corpus["text"], corpus["pid"]))

print(f"전체 passage: {len(flat):,} / 고유 텍스트: {flat['text'].nunique():,}")
print(f"의료 코퍼스: {len(corpus):,}")
print(corpus["host"].value_counts().head(10).to_string())

In [ ]:
MEDICAL_KEYWORDS = [
    "disease", "disorder", "syndrome", "diabetes", "cancer", "asthma", "pneumonia", "infection",
    "arthritis", "stroke", "hypertension", "anemia", "symptom", "pain", "fever", "headache",
    "nausea", "vomiting", "dizziness", "fatigue", "swelling", "bleeding", "diagnosis", "diagnose",
    "blood test", "biopsy", "mri", "ct scan", "x-ray", "ultrasound", "medication", "medicine",
    "drug", "antibiotic", "insulin", "vaccine", "treatment", "therapy", "surgery", "prescription",
    "side effect", "heart", "brain", "kidney", "liver", "lung", "blood pressure", "hormone",
    "immune system", "pregnancy", "depression", "anxiety", "adhd", "mental health", "bipolar",
]
keyword_pattern = re.compile(
    r"(?<!\w)(?:" + "|".join(re.escape(k) for k in MEDICAL_KEYWORDS) + r")(?!\w)",
    flags=re.IGNORECASE,
)

def build_query_sets(split):
    df = qa[split]
    fs = flat[flat["split"] == split]
    gold = (
        fs[fs["is_medical"] & (fs["is_selected"] == 1)]
        .groupby("query_id")["text"]
        .apply(lambda ts: sorted({text2pid[t] for t in ts}))
    )
    any_med = set(fs.loc[fs["is_medical"], "query_id"])

    base = pd.DataFrame({
        "query_id": df["query_id"].astype(int),
        "query": df["query"].map(normalize_text),
        "query_type": df["query_type"],
        "answers": df["answers"].map(list),
    })
    answerable = df["answers"].map(has_answer)

    med = base[answerable & base["query_id"].isin(gold.index)].copy()
    med["gold_pids"] = med["query_id"].map(gold)

    ood = base[
        answerable
        & ~base["query_id"].isin(any_med)
        & ~base["query"].str.contains(keyword_pattern)
    ].copy()
    ood["gold_pids"] = [[] for _ in range(len(ood))]
    return med.reset_index(drop=True), ood.reset_index(drop=True)

med_val, ood_val = build_query_sets("validation")
med_train, ood_train = build_query_sets("train")

qsets = {
    "med_test": med_val,
    "ood_test": ood_val.sample(N_OOD_TEST, random_state=SEED).reset_index(drop=True),
    "med_tune": med_train.sample(N_TUNE, random_state=SEED).reset_index(drop=True),
    "ood_tune": ood_train.sample(N_TUNE, random_state=SEED).reset_index(drop=True),
}
for name, q in qsets.items():
    print(f"{name:9s} {len(q):5d}")
display(qsets["med_test"].head())

In [ ]:
labels = pd.read_parquet(LABEL_PATH)
url_sets = qsets   # 1-1에서 만든 URL 기준 세트
FLAG_COLS = ["label", "high_risk", "time_sensitive"]

def med_with_label(med_df, split, label):
    lab = labels[(labels["split"] == split) & (labels["pool"] == "med_url") & (labels["label"] == label)]
    return med_df.merge(lab[["query_id"] + FLAG_COLS], on="query_id").reset_index(drop=True)

def ood_with_label(split, n):
    lab = labels[(labels["split"] == split) & (labels["pool"] == "no_med") & (labels["label"] == "C")]
    out = lab[["query_id", "query", "query_type", "answers"] + FLAG_COLS].copy()
    out["answers"] = out["answers"].map(list)
    out["gold_pids"] = [[] for _ in range(len(out))]
    return out.sample(min(n, len(out)), random_state=SEED).reset_index(drop=True)

lab_med_train = med_with_label(med_train, "train", "A")
qsets = {
    "url_med_test": url_sets["med_test"],
    "url_med_tune": url_sets["med_tune"],
    "url_ood_tune": url_sets["ood_tune"],
    "lab_med_test": med_with_label(med_val, "validation", "A"),
    "lab_b_test": med_with_label(med_val, "validation", "B"),
    "lab_ood_test": ood_with_label("validation", N_OOD_TEST),
    "lab_med_tune": lab_med_train.sample(min(N_TUNE, len(lab_med_train)), random_state=SEED).reset_index(drop=True),
    "lab_ood_tune": ood_with_label("train", N_TUNE),
}
for name, q in qsets.items():
    print(f"{name:13s} {len(q):5d}")
print("\nvalidation med_url 라벨 분포:")
print(labels[(labels["split"] == "validation") & (labels["pool"] == "med_url")]["label"].value_counts().to_string())

In [ ]:
# 시험(validation) 질문의 라벨은 DB에 반영하지 않음: DB는 질문보다 먼저 만들어지는 것이므로
med_keys = labels.loc[(labels["split"] == "train") & labels["label"].isin(["A", "B"]), ["split", "query_id"]]
in_med_q = pd.MultiIndex.from_frame(flat[["split", "query_id"]]).isin(list(med_keys.itertuples(index=False, name=None)))
extra = (
    flat[in_med_q & ~flat["text"].isin(text2pid)]
    .drop_duplicates("text")[["text", "url", "host"]]
    .reset_index(drop=True)
)
extra.insert(0, "pid", np.arange(len(corpus), len(corpus) + len(extra)))
aug_corpus = pd.concat([corpus, extra], ignore_index=True)   # 행 번호 == pid
aug_text2pid = dict(zip(aug_corpus["text"], aug_corpus["pid"]))

def ext_med_set(split, need_gold):
    """정답이 일반 사이트에만 있는 A 질문. 정답 passage가 보강 DB에 있으면 그 pid를 표시"""
    lab = labels[(labels["split"] == split) & (labels["pool"] == "no_med") & (labels["label"] == "A")]
    fs = flat[(flat["split"] == split) & (flat["is_selected"] == 1)]
    gold = fs.groupby("query_id")["text"].apply(lambda ts: sorted({aug_text2pid[t] for t in ts if t in aug_text2pid}))
    out = lab[["query_id", "query", "query_type", "answers"] + FLAG_COLS].copy()
    out["answers"] = out["answers"].map(list)
    out["gold_pids"] = out["query_id"].map(gold).map(lambda g: g if isinstance(g, list) else [])
    if need_gold:   # 임계값 튜닝용 train 질문은 정답 passage가 DB에 있어야 함
        out = out[out["gold_pids"].map(len) > 0]
    return out.reset_index(drop=True)

qsets["lab_med_ext_test"] = ext_med_set("validation", need_gold=False)
ext_tune = ext_med_set("train", need_gold=True)
qsets["lab_med_ext_tune"] = ext_tune.sample(min(N_TUNE, len(ext_tune)), random_state=SEED).reset_index(drop=True)

print(f"URL DB {len(corpus):,} → 보강 DB {len(aug_corpus):,} (+{len(extra):,})")
print(extra["host"].value_counts().head(10).to_string())
for name in ["lab_med_ext_test", "lab_med_ext_tune"]:
    print(f"{name:17s} {len(qsets[name]):5d}")
in_db = qsets["lab_med_ext_test"]["gold_pids"].map(len) > 0
print(f"lab_med_ext_test 중 정답 passage가 보강 DB에 있는 질문: {in_db.sum()} / {len(in_db)}")

## 2. 검색 (LabeledBaseline과 동일 코드)

In [ ]:
import bm25s, Stemmer

stemmer = Stemmer.Stemmer("english")
N_URL = len(corpus)
passages = aug_corpus["text"].tolist()   # 앞 N_URL개 = URL DB, 전체 = 보강 DB (pid 공유)
DB_SIZE = {"url": N_URL, "aug": len(passages)}

def build_bm25(texts):
    index = bm25s.BM25()
    index.index(bm25s.tokenize(texts, stopwords="en", stemmer=stemmer))
    return index

bm25_index = {db: build_bm25(passages[:n]) for db, n in DB_SIZE.items()}

def bm25_search(queries, db="url", k=K_CAND):
    tokens = bm25s.tokenize(queries, stopwords="en", stemmer=stemmer, show_progress=False)
    ids, scores = bm25_index[db].retrieve(tokens, k=k, show_progress=False)
    return ids, scores

In [ ]:
import torch, faiss
from sentence_transformers import SentenceTransformer

if sys.platform == "darwin":   # Mac: torch와 faiss가 서로 다른 OpenMP를 불러와, faiss가 여러 스레드로 검색하면 커널이 죽음
    faiss.omp_set_num_threads(1)

embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)
if USE_FP16:
    embedder.half()

def encode_passages(texts, path):
    if path.exists():
        return np.load(path)
    emb = embedder.encode(
        texts, batch_size=256, normalize_embeddings=True,
        convert_to_numpy=True, show_progress_bar=True,
    ).astype("float32")
    np.save(path, emb)
    return emb

model_tag = EMBED_MODEL.split("/")[-1]
extra_tag = hashlib.md5("\n".join(passages[N_URL:]).encode()).hexdigest()[:8]   # 라벨이 바뀌면 캐시도 새로
corpus_emb = np.concatenate([
    encode_passages(passages[:N_URL], INDEX_DIR / f"{model_tag}_{N_URL}.npy"),   # MedicalBaseline과 같은 캐시
    encode_passages(passages[N_URL:], INDEX_DIR / f"{model_tag}_aug_{extra_tag}.npy"),
])

faiss_index = {}
for db, n in DB_SIZE.items():
    faiss_index[db] = faiss.IndexFlatIP(corpus_emb.shape[1])   # 정규화된 벡터의 내적 = 코사인 유사도
    faiss_index[db].add(corpus_emb[:n])

def dense_search(queries, db="url", k=K_CAND):
    q_emb = embedder.encode(
        [QUERY_PREFIX + q for q in queries], batch_size=256,
        normalize_embeddings=True, convert_to_numpy=True,
    ).astype("float32")
    scores, ids = faiss_index[db].search(q_emb, k)
    return ids, scores

print(corpus_emb.shape)

In [ ]:
def rrf_fuse(*id_lists_per_run, k=RRF_K, top=K_RERANK):
    """run마다 (n_query, K) 형태의 id 배열을 받아 질의별로 RRF로 합친 상위 top개 pid 리스트를 반환."""
    fused = []
    for per_query in zip(*id_lists_per_run):
        score = defaultdict(float)
        for ids in per_query:
            for rank, pid in enumerate(ids):
                score[int(pid)] += 1.0 / (k + rank + 1)
        fused.append(sorted(score, key=score.get, reverse=True)[:top])
    return fused

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(RERANK_MODEL, device=DEVICE, max_length=512)

def rerank(queries, candidates):
    """candidates: 질의별 pid 리스트 → (정렬된 pid 리스트, 정렬된 점수 리스트)"""
    pairs = [(q, passages[pid]) for q, cand in zip(queries, candidates) for pid in cand]
    flat_scores = reranker.predict(pairs, batch_size=256, show_progress_bar=False)
    out_ids, out_scores, pos = [], [], 0
    for cand in candidates:
        s = np.asarray(flat_scores[pos:pos + len(cand)])
        pos += len(cand)
        order = np.argsort(-s)
        out_ids.append([cand[i] for i in order])
        out_scores.append(s[order].tolist())
    return out_ids, out_scores

In [ ]:
def run_retrieval(queries, db):
    bm25_ids, _ = bm25_search(queries, db)
    dense_ids, _ = dense_search(queries, db)
    hybrid_ids = rrf_fuse(bm25_ids, dense_ids)
    rr_ids, rr_scores = rerank(queries, hybrid_ids)
    return {
        "BM25": [list(map(int, x)) for x in bm25_ids],
        "Dense": [list(map(int, x)) for x in dense_ids],
        "Hybrid": hybrid_ids,
        "Hybrid+Rerank": rr_ids,
    }, rr_scores

In [ ]:
# 거절 기준값: URL baseline과 같은 방법 (URL 튜닝 세트에서 BalancedAcc 최대)
def top1(scores):
    return np.array([s[0] for s in scores])

ret = {}   # ret[세트] = (상위 pid 리스트, 점수 리스트)
for name in ["url_med_tune", "url_ood_tune", "lab_med_test", "lab_med_ext_test", "lab_ood_test"]:
    runs, scores = run_retrieval(qsets[name]["query"].tolist(), GEN_DB)
    ret[name] = (runs["Hybrid+Rerank"], scores)
    print(f"{name} done")

med, ood = top1(ret["url_med_tune"][1]), top1(ret["url_ood_tune"][1])
cand = np.unique(np.concatenate([med, ood]))
bal = [((med >= t).mean() + (ood < t).mean()) / 2 for t in cand]
THR = float(cand[int(np.argmax(bal))])
print(f"THR = {THR:.4f} (tune bal acc {max(bal):.4f})")

## 3. 프롬프트와 채점 (LabeledBaseline과 동일 코드)

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def _tokens(s):
    return re.findall(r"[a-z0-9]+", s.lower())

def is_refusal(pred):
    return "cannot answer" in pred.lower()

def answer_recall(pred, ref):
    ref_t, pred_t = _tokens(ref), set(_tokens(pred))
    return sum(t in pred_t for t in ref_t) / len(ref_t) if ref_t else 0.0

def generation_metrics(preds, answers_list):
    rouge = [max(scorer.score(a, p)["rougeL"].fmeasure for a in ans) for p, ans in zip(preds, answers_list)]
    recall = [max(answer_recall(p, a) for a in ans) for p, ans in zip(preds, answers_list)]
    return {"ROUGE-L": np.mean(rouge), "AnswerRecall": np.mean(recall),
            "RefusalRate": np.mean([is_refusal(p) for p in preds])}

In [ ]:
SYSTEM_CITE = f"""You are a medical information assistant. Answer the question using ONLY the information in the provided passages.
Rules:
1. Do not use any knowledge that is not stated in the passages.
2. Answer in one or two complete sentences that restate the subject of the question.
3. End each sentence with the numbers of the passages that support it, like [1] or [2][3].
4. If the passages do not contain the answer, reply exactly: {REFUSAL}"""

def build_cite_messages(query, pids):
    ctx = "\n\n".join(f"[{i}] {passages[pid]}" for i, pid in enumerate(pids, 1))
    return [{"role": "system", "content": SYSTEM_CITE},
            {"role": "user", "content": f"Passages:\n{ctx}\n\nQuestion: {query}\nAnswer:"}]

CITE_RE = re.compile(r"\[(\d+)\]")
CITE_GROUP_RE = re.compile(r"((?:\s*\[\d+\])+)")
SUPPORT_MIN = 0.5   # 문장 단어 중 그 문장이 인용한 passage들에 있어야 하는 비율
CITE_MIN = 0.3      # 인용 하나가 근거로 인정되려면 문장 단어 중 이만큼이 그 passage에 있어야 함
STOPWORDS = set("""a an the and or but of to in on at for with by from as is are was were be been being it its this that
these those can may might will would should could do does did has have had not no than then so such which who what
when where why how also more most other some any each into about over after before your you they their them he she his her""".split())

def strip_citations(text):
    return re.sub(r"\s*\[\d+\]", "", text).strip()

def content_words(text):
    return {t for t in _tokens(text) if t not in STOPWORDS}

def split_claims(answer):
    """답변을 (문장, 그 문장에 붙은 인용 번호) 묶음으로 나눔. '문장 [1][2].'와 '문장. [1]' 둘 다 처리"""
    parts = CITE_GROUP_RE.split(answer)
    claims = []
    for text, cites in zip(parts[0::2], parts[1::2] + [""]):
        if content_words(text):
            claims.append((text.strip(" ."), [int(n) for n in CITE_RE.findall(cites)]))
    return claims

def validate_answer(answer, pids):
    """인용 유효성과 근거 지원 여부 → (상태, 근거로 남긴 번호, 근거가 아니라 뺀 번호)"""
    if is_refusal(answer):
        return "refused", [], []
    claims = split_claims(answer)
    cited = sorted({n for _, ns in claims for n in ns})
    if not cited:
        return "no_citation", [], []
    if any(n < 1 or n > len(pids) for n in cited):
        return "invalid_citation", [], cited
    status, kept = "supported", set()
    for claim, ns in claims:
        words = content_words(claim)
        cover = {n: words & set(_tokens(passages[pids[n - 1]])) for n in ns}
        union = set().union(*cover.values()) if cover else set()
        if len(union) / len(words) < SUPPORT_MIN:   # 인용이 없거나, 인용한 passage에 내용이 없는 문장
            status = "unsupported"
        kept |= {n for n, c in cover.items() if len(c) / len(words) >= CITE_MIN}
    return status, sorted(kept), sorted(set(cited) - kept)

## 4. 학습 데이터 (train만)
위 기준으로 답변/거절 예시를 만듭니다. 실제 사용 때와 같이 **거절 게이트를 통과한 질문만** 씁니다 (게이트에서 막힌 질문은 LLM이 보지 않음).

In [ ]:
tune_ids = set().union(*(set(qsets[n]["query_id"]) for n in ["url_med_tune", "url_ood_tune", "lab_med_tune", "lab_ood_tune"]))
val_queries = set(qa["validation"]["query"].map(normalize_text).str.lower())
tr = labels[(labels["split"] == "train") & ~labels["query_id"].isin(tune_ids)
            & ~labels["query"].str.lower().isin(val_queries)].copy()
tr["answers"] = tr["answers"].map(lambda a: [x for x in a if x and x != "No Answer Present."])
tr = tr[tr["answers"].map(len) > 0]

med_pool = tr[tr["label"].isin(["A", "B"])].sample(frac=1, random_state=SEED)
ood_pool = tr[(tr["label"] == "C") & (tr["pool"] == "no_med")]   # lab_ood_test와 같은 정의: 의료 사이트에 정답이 없는 비의료 질문
fs = flat[(flat["split"] == "train") & (flat["is_selected"] == 1)]
gold = fs.groupby("query_id")["text"].apply(lambda ts: {text2pid[t] for t in ts if t in text2pid})

def cite_tag(nums):
    return "".join(f"[{n}]" for n in nums)

def make_target(row, ids):
    """정답 표시 문서 → 정답 단어가 들어 있는 문서 → 없으면 거절"""
    ans = row["answers"][0]
    gold_in = [i for i, pid in enumerate(ids, 1) if pid in gold.get(row["query_id"], set())]
    if gold_in:
        return f"{ans.rstrip(' .')} {cite_tag(gold_in)}.", "answer_gold"
    per = [max(answer_recall(passages[pid], a) for a in row["answers"]) for pid in ids]
    ctx = max(answer_recall(" ".join(passages[pid] for pid in ids), a) for a in row["answers"])
    if ctx >= ANSWER_IN_CTX:
        nums = [i for i, r in enumerate(per, 1) if r >= CITE_PASSAGE_MIN] or [int(np.argmax(per)) + 1]
        return f"{ans.rstrip(' .')} {cite_tag(nums)}.", "answer_content"
    return REFUSAL, "refuse_no_content"

examples, need = [], N_TRAIN + N_VALID
# 비의료 질문 중 게이트를 통과한 것 → 거절 (실제로 LLM이 받게 되는 비의료 질문)
runs, scores = run_retrieval(ood_pool["query"].tolist(), GEN_DB)
for (_, row), ids, s in zip(ood_pool.iterrows(), runs["Hybrid+Rerank"], scores):
    if s[0] >= THR:
        examples.append((row["query"], ids[:K_CTX], REFUSAL, "refuse_non_medical"))
# 의료 질문: 게이트를 통과한 것만, 필요한 수가 찰 때까지 1,000개씩 검색
for start in range(0, len(med_pool), 1000):
    if len(examples) >= need:
        break
    chunk = med_pool.iloc[start:start + 1000]
    runs, scores = run_retrieval(chunk["query"].tolist(), GEN_DB)
    for (_, row), ids, s in zip(chunk.iterrows(), runs["Hybrid+Rerank"], scores):
        if s[0] >= THR and len(examples) < need:
            target, kind = make_target(row, ids[:K_CTX])
            examples.append((row["query"], ids[:K_CTX], target, kind))

random.Random(SEED).shuffle(examples)
kinds = pd.Series([k for *_, k in examples]).value_counts()
print(kinds.to_string(), f"\n합계 {len(examples)} (학습 {N_TRAIN} + loss 확인 {N_VALID})")

def to_record(q, ids, target):
    return {"messages": build_cite_messages(q, ids) + [{"role": "assistant", "content": target}]}

for fname, part in [("train.jsonl", examples[:N_TRAIN]), ("valid.jsonl", examples[N_TRAIN:need])]:
    with open(LORA_DATA_DIR / fname, "w") as f:
        for q, ids, target, _ in part:
            f.write(json.dumps(to_record(q, ids, target), ensure_ascii=False) + "\n")
kinds.to_csv(RESULT_DIR / "train_example_kinds.csv")

for q, ids, target, kind in examples[:4]:
    print(f"\n[{kind}] Q: {q}\n  → {target}")

## 5. 평가 입력 (validation)
학습 전에 평가할 질문의 검색 결과를 모두 정해 둡니다. 거절 게이트에서 막힌 질문은 생성하지 않고 거절로 둡니다 (LabeledBaseline과 같음).

In [ ]:
EVAL_SETS = ["lab_med_test", "lab_med_ext_test", "lab_ood_test"]
eval_items = []   # (세트, 질문 번호, 메시지)
for n in EVAL_SETS:
    ids_list, scores = ret[n]
    for i, (q, ids, s) in enumerate(zip(qsets[n]["query"], ids_list, scores)):
        if s[0] >= THR:
            eval_items.append((n, i, build_cite_messages(q, ids[:K_CTX])))
print({n: sum(e[0] == n for e in eval_items) for n in EVAL_SETS}, "개를 생성 (게이트 통과)")

# 학습 동안 메모리를 비우기 위해 검색 모델과 원본 데이터셋을 내림 (이후에는 passages, qsets만 씀)
import gc
eval_passages = {n: [ids[:K_CTX] for ids in ret[n][0]] for n in EVAL_SETS}
for name in ["embedder", "reranker", "bm25_index", "faiss_index", "corpus_emb", "dataset", "qa", "rows", "flat", "fs"]:
    globals().pop(name, None)
gc.collect()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

## 6. LoRA 학습 (MLX)
정답 부분만 loss에 넣고(`--mask-prompt`) 1 epoch 학습합니다. 50단계마다 어댑터를 저장합니다.

In [ ]:
import subprocess, shutil

if RETRAIN:
    shutil.rmtree(LORA_DIR, ignore_errors=True)
    LORA_DIR.mkdir(parents=True)
# 50단계마다 저장된 중간 결과로 진행 단계를 계산 (중간에 멈췄다가 다시 실행하면 거기서부터 이어서 학습)
prog_path = LORA_DIR / "progress.json"
offset = json.loads(prog_path.read_text())["done_iters"] if prog_path.exists() else 0
ckpts = sorted(LORA_DIR.glob("[0-9]*_adapters.safetensors"))
done_iters = offset + (int(ckpts[-1].name.split("_")[0]) if ckpts else 0)

if done_iters >= LORA_ITERS:
    print("학습 완료된 LoRA 사용:", LORA_DIR)
else:
    cmd = [sys.executable, "-m", "mlx_lm", "lora", "--model", MLX_MODEL, "--train",
           "--data", str(LORA_DATA_DIR), "--adapter-path", str(LORA_DIR), "--mask-prompt", "--grad-checkpoint",
           "--batch-size", str(LORA_BATCH), "--grad-accumulation-steps", str(LORA_GRAD_ACC),
           "--iters", str(LORA_ITERS - done_iters), "--learning-rate", str(LORA_LR), "--num-layers", str(LORA_LAYERS),
           "--max-seq-length", str(LORA_MAX_LEN), "--steps-per-report", "20", "--steps-per-eval", "200",
           "--val-batches", "25", "--save-every", "50", "--seed", str(SEED)]
    if done_iters:
        # 다시 시작하면 mlx_lm의 단계 번호가 1부터 새로 매겨지므로, 이어받을 파일과 지금까지의 단계 수를 따로 기록
        resume = LORA_DIR / "resume_from.safetensors"
        if ckpts:
            shutil.copy(ckpts[-1], resume)
            for c in ckpts:
                c.unlink()
        prog_path.write_text(json.dumps({"done_iters": done_iters}))
        cmd += ["--resume-adapter-file", str(resume)]
        print(f"{done_iters}/{LORA_ITERS}단계부터 이어서 학습 (학습 데이터는 처음 순서부터 다시 읽음)")
    with open(RESULT_DIR / "train_log.txt", "a") as log:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:
            log.write(line); log.flush()
            if any(k in line for k in ("Iter", "loss", "Error", "error", "Saved")):
                print(line.rstrip(), flush=True)
    assert proc.wait() == 0, "학습 실패: results/lora/train_log.txt 확인"

## 7. 원본 vs LoRA 생성 (validation)
두 모델 모두 같은 MLX 4bit 모델로, 같은 입력을 그리디로 생성합니다. 결과는 `gen_cache_mlx.jsonl`에 바로 저장해 중간에 멈춰도 이어서 진행합니다.

In [ ]:
from mlx_lm import load, batch_generate
from tqdm.auto import tqdm

CACHE_PATH = RESULT_DIR / "gen_cache_mlx.jsonl"
cache = {}
if CACHE_PATH.exists():
    with open(CACHE_PATH) as f:
        for line in f:
            rec = json.loads(line)
            cache[rec["key"]] = rec["output"]

def gen_key(tag, messages):
    return hashlib.md5(json.dumps([MLX_MODEL, tag, MAX_NEW_TOKENS, messages]).encode()).hexdigest()

def mlx_generate(tag, adapter_path=None, save_every=64):
    keys = [gen_key(tag, m) for _, _, m in eval_items]
    todo = [(k, m) for k, (_, _, m) in zip(keys, eval_items) if k not in cache]
    print(f"[{tag}] 저장된 답변 {len(keys) - len(todo)}, 새로 생성 {len(todo)}")
    if todo:
        model, tok = load(MLX_MODEL, adapter_path=adapter_path)
        with open(CACHE_PATH, "a") as f, tqdm(total=len(todo)) as bar:
            for s in range(0, len(todo), save_every):
                chunk = todo[s:s + save_every]
                for b in range(0, len(chunk), EVAL_BATCH):
                    part = chunk[b:b + EVAL_BATCH]
                    prompts = [tok.apply_chat_template(m, add_generation_prompt=True) for _, m in part]
                    outs = batch_generate(model, tok, prompts, max_tokens=MAX_NEW_TOKENS).texts
                    for (k, _), o in zip(part, outs):
                        cache[k] = o.strip()
                        f.write(json.dumps({"key": k, "output": cache[k]}, ensure_ascii=False) + "\n")
                    bar.update(len(part))
                f.flush()
        del model, tok
        gc.collect()
    return [cache[k] for k in keys]

gen = {"원본 Qwen": mlx_generate("base"), "LoRA Qwen": mlx_generate("lora", str(LORA_DIR))}

## 8. 비교
지표는 `LabeledBaseline.ipynb` 6-3과 같습니다. 점수는 인용 번호를 지우고 계산합니다. 게이트에서 막힌 질문은 두 모델 모두 거절입니다.

In [ ]:
def assemble(outputs):
    """세트별 전체 답변 (게이트에서 막힌 질문은 거절)"""
    ans = {n: [REFUSAL] * len(qsets[n]) for n in EVAL_SETS}
    for (n, i, _), o in zip(eval_items, outputs):
        ans[n][i] = o
    return ans

rows, status_rows, answers = {}, {}, {}
for name, outs in gen.items():
    ans = assemble(outs)
    answers[name] = ans
    row = {}
    for n, tag in [("lab_med_test", "medsite"), ("lab_med_ext_test", "extsite")]:
        m = generation_metrics([strip_citations(a) for a in ans[n]], qsets[n]["answers"].tolist())
        row[f"Refusal_{tag}"], row[f"AnswerRecall_{tag}"] = m["RefusalRate"], m["AnswerRecall"]
        if tag == "medsite":
            row["ROUGE-L"] = m["ROUGE-L"]
    row["OODRefusal"] = np.mean([is_refusal(a) for a in ans["lab_ood_test"]])
    row["BalancedAcc"] = (1 - row["Refusal_medsite"] + row["OODRefusal"]) / 2
    answered = [a for a in ans["lab_med_test"] if not is_refusal(a)]
    row["AvgWords"] = np.mean([len(strip_citations(a).split()) for a in answered])
    rows[name] = row
    status_rows[name] = pd.Series([validate_answer(a, ids)[0] for a, ids in
                                   zip(ans["lab_med_test"], eval_passages["lab_med_test"])]).value_counts()

cols = ["Refusal_medsite", "OODRefusal", "BalancedAcc", "ROUGE-L", "AnswerRecall_medsite",
        "Refusal_extsite", "AnswerRecall_extsite", "AvgWords"]
lora_table = pd.DataFrame(rows).T[cols].round(4)
lora_status = pd.DataFrame(status_rows).fillna(0).astype(int)
lora_table.to_csv(RESULT_DIR / "lora_comparison.csv")
lora_status.to_csv(RESULT_DIR / "lora_citation_status.csv")
print(f"THR = {THR:.4f} / 학습 예시 {N_TRAIN} / 검증 상태 (lab_med_test):")
display(lora_status)
lora_table

In [ ]:
# 사례: 원본은 답했는데 LoRA가 거절한 의료 질문 (잘못 거절이 늘었는지 확인)
cmp = qsets["lab_med_test"][["query", "answers"]].copy()
for name in gen:
    cmp[name] = answers[name]["lab_med_test"]
cmp.to_json(RESULT_DIR / "lab_med_test_predictions.jsonl", orient="records", lines=True, force_ascii=False)
pd.set_option("display.max_colwidth", 200)
newly = cmp[~cmp["원본 Qwen"].map(is_refusal) & cmp["LoRA Qwen"].map(is_refusal)]
print(f"원본은 답했는데 LoRA가 거절: {len(newly)} / 반대: {(cmp['원본 Qwen'].map(is_refusal) & ~cmp['LoRA Qwen'].map(is_refusal)).sum()}")
display(newly.head(10))
display(cmp.sample(5, random_state=SEED))